# Impulse and Torque MLP (vertex-transform, no GNN)

This notebook trains a physics-structured and physics-informed MLP for a
rigid body interacting with a plane.

**Variant:** the GCN / GNN is removed. We still place the body's mesh
vertices in world space (rotation + translation, plus per-vertex velocity
from rigid-body kinematics), but those transformed vertices are flattened
and fed **directly into the MLP** — no graph convolutions, no `edge_index`,
no `torch_geometric`.

As input we get the rotation as a 6-D rotation matrix (with sin and cos
of roll / pitch / yaw). We parameterise normal force with Hooke, similar
to our simulations. We internally predict the contact normal. We couple
torque to force via cross product: `torque = r_lever x f`. In the loss we
use a Huber loss with a weight for energy conservation.


This notebook trains a physics-structured and physics informed MLP for a rigid body interacting with a plane-
As input we get the rotation as a 6-D rotation Matrix (with cos and sin)
We parameterise normal force with Hooke, similar to our simulations
We internally predict the contact normal
We couple torque to force via cross product: torque = r_{lever} x f
In the loss we use a Hube loss with a weight for energy conservation

In [1]:
%pip install trimesh
%pip install fast-simplification

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 12.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.1 MB/s eta 0:00:0000:010:01


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import json
import math
from tqdm import tqdm
import trimesh

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

Using device: cuda


## Constants

In [ ]:
TRAIN_TEST_SPLIT = 0.8

## Colab Only - Download data

In [5]:
def is_colab():
    try:
        import google.colab
        return True
    except Exception as e:
        return False

if is_colab():
    from google.colab import drive
    from tqdm import tqdm
    import os
    import json
    import shutil

    drive.mount('/content/drive', force_remount=True)

    # --- Copy the JSON file ---
    src_path = "/content/drive/MyDrive/final_output_contact_points.json"
    dst_path = "/content/final_output_contact_points.json"
    chunk_size = 1024 * 1024  # 1 MB
    file_size = os.path.getsize(src_path)

    with open(src_path, 'rb') as src, open(dst_path, 'wb') as dst:
        with tqdm(total=file_size, unit='B', unit_scale=True, desc="Copying JSON to /content") as pbar:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))
    print("Done! File is now in:", dst_path)

    # --- Copy the blender_models folder ---
    src_folder = "/content/drive/MyDrive/blender_models"
    dst_folder = "/content/blender_models"

    # Gather all files first so we know the total size for the progress bar
    all_files = []
    total_size = 0
    for root, dirs, files in os.walk(src_folder):
        for f in files:
            full = os.path.join(root, f)
            try:
                size = os.path.getsize(full)
            except OSError:
                size = 0
            all_files.append((full, size))
            total_size += size

    print(f"\nFound {len(all_files)} files in blender_models ({total_size / (1024**2):.1f} MB total)")

    os.makedirs(dst_folder, exist_ok=True)

    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Copying blender_models") as pbar:
        for src_file, size in all_files:
            rel = os.path.relpath(src_file, src_folder)
            dst_file = os.path.join(dst_folder, rel)
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)

            with open(src_file, 'rb') as src, open(dst_file, 'wb') as dst:
                while True:
                    chunk = src.read(chunk_size)
                    if not chunk:
                        break
                    dst.write(chunk)
                    pbar.update(len(chunk))

    print("Done! Folder is now in:", dst_folder)

    # --- Print the first entry of the JSON ---
    with open(dst_path, 'r') as f:
        data = json.load(f)
    first = data[0] if isinstance(data, list) else next(iter(data.values()))
    print("\nFirst entry:")
    print(json.dumps(first, indent=2))

Mounted at /content/drive


Copying JSON to /content: 100%|██████████| 3.16G/3.16G [00:32<00:00, 96.0MB/s]


Done! File is now in: /content/final_output_contact_points.json

Found 10 files in blender_models (23.7 MB total)


Copying blender_models: 100%|██████████| 24.9M/24.9M [00:04<00:00, 5.19MB/s]


Done! Folder is now in: /content/blender_models

First entry:
{
  "self_position": {
    "x": -1.3479015898986337e-10,
    "y": -9.26035481738977e-11,
    "z": 0.3049439538719369
  },
  "linear_velocity": {
    "x": -5.75398472817878e-11,
    "y": 5.826339211869911e-11,
    "z": 0.03890375386409823
  },
  "angular_velocity": {
    "x": -0.007683846272249534,
    "y": 0.08600533370751791,
    "z": 0.019988591559377294
  },
  "self_rotation": {
    "qx": 0.9667131061933467,
    "qy": -0.16328529123739366,
    "qz": -0.07749282570235115,
    "qw": -0.1811036883782211,
    "roll": -2.8029813988476615,
    "pitch": 0.2105213802444965,
    "yaw": -0.29854016498091274
  },
  "collider_position": {
    "x": 0.0,
    "y": 0.0,
    "z": 0.0
  },
  "collider_rotation": {
    "qx": 0.0,
    "qy": 0.0,
    "qz": 0.0,
    "qw": 1.0,
    "roll": 0.0,
    "pitch": -0.0,
    "yaw": 0.0
  },
  "relative_position_to_collider": {
    "x": -1.3479015898986337e-10,
    "y": -9.26035481738977e-11,
    "z": 0

## Dataset

Here we get the contat points with individual forces. From these forces, we calculate the torque and sum up the forces and torques to one force and one torque

In [6]:
class ContactDataset(Dataset):
    """World-frame wrench dataset for a rigid body on a plane.

    Features (13-D):
        [v_x, v_y, v_z,                           # linear velocity (world frame)
         w_x, w_y, w_z,                           # angular velocity (world frame)
         rel_pos_z,                               # height above plane
         sin(roll), cos(roll),
         sin(pitch), cos(pitch),
         sin(yaw), cos(yaw)]

    Per-sample extras (used by the GCN to place vertices in world space):
        self_position: (3,) world-frame position of the body origin (= COM
                       used in the torque computation: lever = r_world - cube_pos).

    Targets (world frame, PHYSICAL UNITS — not normalised):
        force:  (3,)   sum of per-contact forces
        torque: (3,)   sum of (r_world - com_world) x f_world
    """

    def __init__(self, data_list):
        features, forces, torques, collisions = [], [], [], []
        lin_vels, ang_vels, self_positions = [], [], []

        for contact in data_list:
            rel_pos = contact["relative_position_to_collider"]
            rel_rot = contact["relative_rotation_to_collider"]
            lin_vel = contact["linear_velocity"]
            ang_vel = contact["angular_velocity"]
            cube_pos = contact["self_position"]

            roll, pitch, yaw = rel_rot["roll"], rel_rot["pitch"], rel_rot["yaw"]
            feats = np.array([
                lin_vel["x"], lin_vel["y"], lin_vel["z"],
                ang_vel["x"], ang_vel["y"], ang_vel["z"],
                rel_pos["z"],
                np.sin(roll), np.cos(roll),
                np.sin(pitch), np.cos(pitch),
                np.sin(yaw), np.cos(yaw),
            ], dtype=np.float32)

            cube_pos_numpy = np.array([cube_pos["x"], cube_pos["y"], cube_pos["z"]],
                                      dtype=np.float32)
            force_numpy = np.zeros(3, dtype=np.float32)
            torque_numpy = np.zeros(3, dtype=np.float32)

            for p in contact.get("points", []):
                p_force = p["force"]
                lever_pos = p["contact_position_world"]
                point_force_numpy = np.array(
                    [p_force["x"], p_force["y"], p_force["z"]], dtype=np.float32)
                lever_rel_pos = np.array(
                    [lever_pos["x"], lever_pos["y"], lever_pos["z"]],
                    dtype=np.float32) - cube_pos_numpy

                force_numpy += point_force_numpy
                torque_numpy += np.cross(lever_rel_pos, point_force_numpy)

            features.append(feats)
            forces.append(force_numpy)
            torques.append(torque_numpy)
            collisions.append([min(len(contact.get("points", [])), 1)])
            lin_vels.append([lin_vel["x"], lin_vel["y"], lin_vel["z"]])
            ang_vels.append([ang_vel["x"], ang_vel["y"], ang_vel["z"]])
            self_positions.append(cube_pos_numpy)

        self.features       = torch.FloatTensor(np.asarray(features))
        self.forces         = torch.FloatTensor(np.asarray(forces))
        self.torques        = torch.FloatTensor(np.asarray(torques))
        self.collisions     = torch.FloatTensor(np.asarray(collisions))
        self.lin_vels       = torch.FloatTensor(np.asarray(lin_vels, dtype=np.float32))
        self.ang_vels       = torch.FloatTensor(np.asarray(ang_vels, dtype=np.float32))
        self.self_positions = torch.FloatTensor(np.asarray(self_positions, dtype=np.float32))

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (
            self.features[idx],
            {
                "force":         self.forces[idx],
                "torque":        self.torques[idx],
                "is_collision":  self.collisions[idx],
                "lin_vel":       self.lin_vels[idx],
                "ang_vel":       self.ang_vels[idx],
                "self_position": self.self_positions[idx],
            },
        )

### Show dataset

show the first entry of the dataset

In [7]:
json_file = 'final_output_contact_points.json'
with open(json_file, 'r') as f:
    data = json.load(f)
if isinstance(data, dict):
    data = [data]

full_dataset = ContactDataset(data)


Print the first entry of the dataset

In [8]:
print(full_dataset[0])

(tensor([-5.7540e-11,  5.8263e-11,  3.8904e-02, -7.6838e-03,  8.6005e-02,
         1.9989e-02,  3.0494e-01, -3.3218e-01, -9.4322e-01,  2.0897e-01,
         9.7792e-01, -2.9413e-01,  9.5577e-01]), {'force': tensor([-9.1224e-13,  6.0816e-12,  2.8818e-01]), 'torque': tensor([ 8.8943e-03, -3.1792e-02,  6.9908e-13]), 'is_collision': tensor([1.]), 'lin_vel': tensor([-5.7540e-11,  5.8263e-11,  3.8904e-02]), 'ang_vel': tensor([-0.0077,  0.0860,  0.0200]), 'self_position': tensor([-1.3479e-10, -9.2604e-11,  3.0494e-01])})


Showing some info of the dataset

In [9]:
print("collisions:", int(full_dataset.collisions.sum().item()),
      "/", len(full_dataset))
mask = full_dataset.collisions.squeeze(-1).bool()
if mask.any():
    f_rms = full_dataset.forces[mask].pow(2).mean().sqrt().item()
    t_rms = full_dataset.torques[mask].pow(2).mean().sqrt().item()
    print(f"\nContact-only RMS force  = {f_rms:.4f}")
    print(f"Contact-only RMS torque = {t_rms:.4f}")
    print(f"Suggested w_torque / w_force ratio ≈ {f_rms / max(t_rms, 1e-8):.3f}")


collisions: 1024037 / 1958062

Contact-only RMS force  = 1064.2872
Contact-only RMS torque = 202.8762
Suggested w_torque / w_force ratio ≈ 5.246


Split the dataset into train and test dataset and create data loaders

In [10]:
train_size = int(TRAIN_TEST_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator = torch.Generator())

use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=4 if use_gpu else 0,
    pin_memory=use_gpu,
    persistent_workers=use_gpu,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=4 if use_gpu else 0,
    pin_memory=use_gpu,
    persistent_workers=use_gpu,
)

Validation checks on the dataset:

In [11]:
input_dim = full_dataset.features.shape[1]
print(f"input_dim={input_dim}  (expect 10)")
print(f"force target shape = {tuple(full_dataset.forces.shape)}  (expect (N, 3))")
print(f"torque target shape = {tuple(full_dataset.torques.shape)}  (expect (N, 3))")
assert max(full_dataset.collisions) == 1


input_dim=13  (expect 10)
force target shape = (1958062, 3)  (expect (N, 3))
torque target shape = (1958062, 3)  (expect (N, 3))


## Mesh (local-frame vertices)

We load the mesh and keep only the **vertex positions in the body's local
frame**. There is no graph here — no edges, no neighbours — because the
MLP doesn't operate on a graph. The vertices are a fixed-size, fixed-order
geometric description of the body's shape, and we feed them into the MLP
after rotating them into world space.


In [12]:
import trimesh
import numpy as np

# We still need the mesh: its vertices (in the local body frame) are the
# geometric input to the MLP after the world-space transform.
mesh = trimesh.load('blender_models/bunny.obj',
                    process=True, force='mesh')
mesh.merge_vertices(merge_tex=True, merge_norm=True)

mesh = mesh.simplify_quadric_decimation(face_count=1000)

print(mesh.vertices.shape)

vertice_positions = np.array(mesh.vertices)
num_vertices = len(mesh.vertices)


(502, 3)


## Model

Physically structured MLP. The vertices are rotated (and translated) into
world space, the per-vertex world-frame velocity is computed, and then
**everything is flattened and fed into the MLP** — no graph convolutions,
no message passing.


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

HEAD_OUT_DIM = 11  # 1 (collision) + 1 (depth) + 3 (force_residual) + 3 (normal) + 3 (lever)
VEL_SLICE = slice(0, 3)  # [vx, vy, vz] are the first 3 features


class ResBlock(nn.Module):
    def __init__(self, width, expansion=4):
        super().__init__()
        self.act = nn.ReLU(inplace=True)

        self.block = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, width * expansion),
            self.act,
            nn.LayerNorm(width * expansion),
            nn.Linear(width * expansion, width),
            self.act,
        )

    def forward(self, x):
        return self.act(x + self.block(x))


class WrenchPredictor(nn.Module):
    """
    Predicts net wrench (force + torque) with a head whose force assembly
    matches the physics sim, plus a learned residual to absorb deviations
    the analytical model cannot express.

    Sim-matching part:
        F_spring  = -k * depth                        (depth <= 0)
        F_damping = -c * (v . n),  c = 2*sqrt(k*m)*b
        F_mag     = max(F_spring + F_damping, 0)      (hard clamp, no softplus)
        F_normal  = F_mag * n                         (along contact normal)

    Residual:
        F_vec     = F_normal + force_residual         (3-vec correction, unconstrained)
        T_vec     = lever x F_vec

    NOTE: `input_dim` here is the *full* feature size going into the MLP,
    which in this variant is the per-sample state (13) PLUS the flattened
    per-vertex world-space features. `forward` still slices the linear
    velocity out of the FIRST 3 entries, which we keep as v_lin convention
    so the damping term is well-defined.
    """

    def __init__(self, input_dim=13, width=256, num_blocks=5,
                 baseline_k=1e3, learn_k=True,
                 baseline_bounciness=0.5, learn_bounciness=True,
                 baseline_mass=1.0, learn_mass=True,
                 head_hidden=64):
        super().__init__()

        # --- backbone ---
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.LayerNorm(width),
            nn.GELU(),
        )

        backbone_layers = [nn.LayerNorm(width)]
        for _ in range(num_blocks, 1, -1):
            backbone_layers.append(ResBlock(width))
        self.backbone = nn.Sequential(*backbone_layers)

        self.head_trunk = nn.Sequential(
            nn.Linear(width, head_hidden),
            nn.LayerNorm(head_hidden),
            nn.GELU(),
        )
        self.head_out = nn.Linear(head_hidden, HEAD_OUT_DIM)

        # Physics parameters.
        self.k = nn.Parameter(
            torch.tensor(baseline_k, dtype=torch.float32),
            requires_grad=learn_k,
        )
        self.bounciness = nn.Parameter(
            torch.tensor(baseline_bounciness, dtype=torch.float32),
            requires_grad=learn_bounciness,
        )
        self.mass = nn.Parameter(
            torch.tensor(baseline_mass, dtype=torch.float32),
            requires_grad=learn_mass,
        )

    def forward(self, x, velocity):
        """
        Args:
            x:        [B, input_dim]  full feature vector going into the MLP
                                       (sample state + flattened vertex features).
            velocity: [B, 3]           body linear velocity in world frame, used
                                       for the Hooke damping term. Passed in
                                       explicitly so the slicing convention is
                                       robust to whatever `x` actually contains.
        """
        h = self.input_proj(x)
        h = self.backbone(h)

        raw = self.head_out(self.head_trunk(h))
        collision_logit, depth_raw, force_residual, normal_raw, lever = raw.split(
            [1, 1, 3, 3, 3], dim=-1
        )

        # Match sim: mass clamped to >= 1e-6, k positive.
        k_pos    = F.softplus(self.k)     if self.k.requires_grad else self.k
        mass_pos = F.softplus(self.mass)  if self.mass.requires_grad else self.mass
        mass_pos = torch.clamp(mass_pos, min=1e-6)
        bounciness = torch.sigmoid(self.bounciness)

        # Critical damping
        c = 2.0 * torch.sqrt(k_pos * mass_pos) * bounciness

        # Match sim's `min(penetration, 0.0)`: allow exact zero, clamp positives.
        depth = torch.clamp(depth_raw, max=0.0)

        # Normalize the contact normal (sim does the same defensively).
        contact_normal = F.normalize(normal_raw, dim=-1, eps=1e-8)

        F_spring = -k_pos * depth

        # Damping force along the normal.
        vel_normal = (velocity * contact_normal).sum(dim=-1, keepdim=True)
        F_damping = -c * vel_normal

        # Sim uses a hard clamp at 0 — never sucks objects into surfaces.
        F_mag = F.relu(F_spring + F_damping)

        # Sim-matching normal force, plus learned residual correction.
        force_normal = F_mag * contact_normal
        force_vec    = force_normal + force_residual
        torque_vec   = torch.cross(lever, force_vec, dim=-1)

        return {
            "collision_logit": collision_logit,
            "force":  force_vec,
            "torque": torque_vec,
            "aux": {
                "contact_normal": contact_normal,
                "lever":          lever,
                "depth":          depth,
                "k":              k_pos.detach(),
                "c":              c.detach(),
                "mass":           mass_pos.detach(),
                "F_mag":          F_mag,
                "F_spring":       F_spring,
                "F_damping":      F_damping,
                "force_normal":   force_normal,
                "force_residual": force_residual,
            },
        }


# --------------------------------------------------------------------------
# World-space vertex transform
# --------------------------------------------------------------------------
# Feature layout (13-D, set by ContactDataset):
#   0:3   linear velocity     v_lin       (world frame)
#   3:6   angular velocity    omega       (world frame)
#   6     rel_pos_z           (height above plane)
#   7:9   (sin roll,  cos roll)
#   9:11  (sin pitch, cos pitch)
#   11:13 (sin yaw,   cos yaw)
#
# Rotation matrix is built directly from sin/cos pairs — no atan2 round-trip
# needed. Convention: intrinsic Z-Y-X (yaw, then pitch, then roll), i.e.
#     R = Rz(yaw) @ Ry(pitch) @ Rx(roll)
# which is what Blender exports for X-Y-Z Euler angles (the most common
# default). If the sim uses a different order, change `_rotmat_from_sincos`.
LIN_VEL_SLICE = slice(0, 3)
ANG_VEL_SLICE = slice(3, 6)
ROLL_SC_SLICE  = slice(7, 9)    # (sin, cos)
PITCH_SC_SLICE = slice(9, 11)
YAW_SC_SLICE   = slice(11, 13)


def _rotmat_from_sincos(features: torch.Tensor) -> torch.Tensor:
    """[B, F] feature batch -> [B, 3, 3] rotation matrix.

    Z-Y-X intrinsic: R = Rz(yaw) Ry(pitch) Rx(roll).
    """
    sr, cr = features[:, 7:8],  features[:, 8:9]
    sp, cp = features[:, 9:10], features[:, 10:11]
    sy, cy = features[:, 11:12], features[:, 12:13]

    # Each row is a [B, 3] tensor; stack along dim=1 -> [B, 3, 3].
    row0 = torch.cat([cy * cp,             cy * sp * sr - sy * cr,  cy * sp * cr + sy * sr], dim=-1)
    row1 = torch.cat([sy * cp,             sy * sp * sr + cy * cr,  sy * sp * cr - cy * sr], dim=-1)
    row2 = torch.cat([-sp,                 cp * sr,                 cp * cr               ], dim=-1)
    return torch.stack([row0, row1, row2], dim=1)


class VertexMLPWrench(nn.Module):
    """
    Vertex-transform MLP (no GNN).

    For each sample we:
      1. Build the body's rotation matrix from the sin/cos features.
      2. Rotate each local vertex into world space, optionally translating
         by `body_position`.
      3. Compute the per-vertex world-frame velocity v_lin + omega x r_world
         (the actual physical velocity at that vertex on the rigid body).
      4. Concatenate, per vertex:
           local_xyz (3) + world_xyz (3) + world_z (1) + vel_at_vertex (3)  = 10
         and FLATTEN across all N vertices into a single fixed-size vector
         of length N * 10. This is concatenated with the original per-sample
         state (13-D) and fed into `WrenchPredictor`.

    There is no message passing and no notion of mesh edges in this model —
    the MLP sees the vertices as a fixed-order positional feature bank, and
    learns whatever spatial structure it needs to from training data.
    """
    # local(3) + world(3) + world_z(1) + vel_at_vertex(3) = 10 per vertex
    VERT_GEOM_DIM = 10

    def __init__(self, state_dim: int = 13,
                 nodes_per_graph: int = num_vertices,
                 width: int = 256, num_blocks: int = 5):
        super().__init__()
        self.N = nodes_per_graph
        self.state_dim = state_dim
        flat_vert_dim = self.N * self.VERT_GEOM_DIM
        self.head = WrenchPredictor(
            input_dim=state_dim + flat_vert_dim,
            width=width, num_blocks=num_blocks,
        )

    def forward(self, features, vertex_pos, body_position=None):
        """
        Args:
            features:      [B, state_dim] per-sample state vector
                           (the 13-D ContactDataset feature).
            vertex_pos:    [N, 3] local-frame OBJ vertex positions
                           (shared across the batch — the rest pose).
            body_position: [B, 3] world-frame position of each body's
                           origin (= `self_position` in the dataset, the
                           same point the torque target was computed about).
                           If None, no translation is applied (body at origin).

        Returns:
            WrenchPredictor output dict.
        """
        B = features.size(0)
        N = self.N

        # --- Per-sample rotation matrix from the sin/cos features ---
        R = _rotmat_from_sincos(features)               # [B, 3, 3]

        # --- Place local vertices in world space ---
        vp_local = vertex_pos.unsqueeze(0).expand(B, N, 3)  # [B, N, 3]

        # World rotation: for each b, n: world_n_i = sum_j R[b, i, j] * vp_local[b, n, j]
        vp_world = torch.einsum("bij,bnj->bni", R, vp_local)  # [B, N, 3]

        if body_position is not None:
            vp_world = vp_world + body_position.unsqueeze(1)  # broadcast over N

        # --- Velocity at each vertex: v_at_vertex = v_lin + omega x r_world ---
        # r_world here is the lever from the body origin: rotated, NOT translated
        # (so this lives in the same frame as the torque target's lever).
        v_lin   = features[:, LIN_VEL_SLICE].unsqueeze(1)   # [B, 1, 3]
        omega   = features[:, ANG_VEL_SLICE].unsqueeze(1)   # [B, 1, 3]
        r_world = torch.einsum("bij,bnj->bni", R, vp_local) # [B, N, 3]
        vel_at_vertex = v_lin + torch.cross(
            omega.expand_as(r_world), r_world, dim=-1)      # [B, N, 3]

        # --- Per-vertex geometric features, stacked then flattened ---
        # [B, N, 10] -> [B, N*10]
        vert_feats = torch.cat([
            vp_local,                # local xyz                 (3)
            vp_world,                # world xyz                 (3)
            vp_world[..., 2:3],      # explicit height           (1)
            vel_at_vertex,           # per-vertex world velocity (3)
        ], dim=-1)                                            # [B, N, 10]
        vert_flat = vert_feats.reshape(B, N * self.VERT_GEOM_DIM)

        # Concat original state + flattened vertex bank -> single MLP input.
        x = torch.cat([features, vert_flat], dim=-1)          # [B, state_dim + N*10]

        # Pass the body's linear velocity through explicitly so the Hooke
        # damping term in the head doesn't depend on the layout of `x`.
        return self.head(x, velocity=features[:, LIN_VEL_SLICE])


def make_fast_predictor(input_dim=13, width=512, num_blocks=5):
    model = VertexMLPWrench(
        state_dim=input_dim,
        nodes_per_graph=num_vertices,
        width=width, num_blocks=num_blocks,
    )
    return model


In [14]:
model = make_fast_predictor().to(device)

Print the model

In [15]:
print(f"Model: {model}")
print(f"Backbone: {model.head.backbone}")
print(f"Head trunk: {model.head.head_trunk}")
print(f"Head out: {model.head.head_out}")

Model: VertexMLPWrench(
  (head): WrenchPredictor(
    (input_proj): Sequential(
      (0): Linear(in_features=5033, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
    )
    (backbone): Sequential(
      (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (1): ResBlock(
        (act): ReLU(inplace=True)
        (block): Sequential(
          (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=512, out_features=2048, bias=True)
          (2): ReLU(inplace=True)
          (3): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (4): Linear(in_features=2048, out_features=512, bias=True)
          (5): ReLU(inplace=True)
        )
      )
      (2): ResBlock(
        (act): ReLU(inplace=True)
        (block): Sequential(
          (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=512, out_features=20

Check the model

In [16]:
model.eval()
with torch.no_grad():
    # Single-sample sanity check: 1 sample of 13-D state.
    features = torch.zeros([1, 13], device=device)
    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)
    out = model(features, vertex_pos=vp)
    print("force shape :", tuple(out["force"].shape))
    print("torque shape:", tuple(out["torque"].shape))


force shape : (1, 3)
torque shape: (1, 3)


### Benchmarking

Benchmark the evaluation speed of the model

In [17]:
NUM_BENCHMARKS = 1000
import time
import torch._logging
torch._logging.set_logs(recompiles=True, graph_breaks=True)

features = torch.rand((1, 13), device=device)
vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

results = []
benchmark_model = make_fast_predictor().to(device).eval()
benchmark_model = torch.compile(benchmark_model)
benchmark_model.eval()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

    with torch.inference_mode():
        # warmup
        for _ in tqdm(range(1000), desc="Warmup"):
            benchmark_model(features, vertex_pos=vp)
        print("=====Warmup completed======")

        for _ in tqdm(range(NUM_BENCHMARKS), desc="Benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            benchmark_model(features, vertex_pos=vp)
            torch.cuda.synchronize()
            end = time.perf_counter()
            results.append((end - start) * 1000)

    torch.backends.cudnn.benchmark = False
    per_pred_ms = sum(results) / NUM_BENCHMARKS
    print(f"Per prediction: {per_pred_ms:.2f} ms")


Warmup: 100%|██████████| 1000/1000 [00:08<00:00, 112.48it/s]


=====Warmup completed======


Benchmark: 100%|██████████| 1000/1000 [00:00<00:00, 1103.07it/s]

Per prediction: 0.89 ms


In [18]:
if torch.cuda.is_available():
    print(f"\n=== Benchmark Results ({device}) ===")
    print(f"Max: {max(results)} ms")
    print(f"Min: {min(results)} ms")
    print(f"Avg: {sum(results) / len(results)} ms")
    print(f"Median: {sorted(results)[len(results) // 2]} ms")
    print("Results:")
    print(results)



=== Benchmark Results (cuda) ===
Max: 2.0429730000159907 ms
Min: 0.7380380000086006 ms
Avg: 0.8855215100054465 ms
Median: 0.8748079999350011 ms
Results:
[2.0429730000159907, 0.9268259998407302, 0.8871750001162582, 0.9042619999490853, 0.9012880000227597, 0.8734539999295521, 0.8601500001077511, 0.8295519999137468, 0.8193590001610573, 0.8909300001960219, 0.8726829998977337, 0.881475000142018, 0.8819930001209286, 0.7965910001530574, 0.7698120000441122, 0.8758690000831848, 0.8700340001723816, 0.8864390001690481, 0.8053399999425892, 0.80444099990018, 0.8820499999728781, 0.877768999998807, 0.8543840001493663, 0.9168369999770221, 0.8530410000275879, 0.8964420001120743, 0.8634609998807719, 0.8262250000825588, 0.7593729999371135, 0.7716990000972146, 0.8874200000263954, 0.8638469998913934, 0.8555179999802931, 0.8496589998685522, 0.8462220000637899, 0.8776450001732883, 0.8722659999875759, 0.8602959999279847, 0.7841159999770753, 0.7998000000952743, 0.7880779999140941, 0.8620589999281947, 0.8649009

### CUDA Graph Benchmark

Captures the forward pass as a CUDA graph so the ~40 per-kernel dispatches are replaced by a single `cuGraphLaunch` on every call.

In [19]:
if torch.cuda.is_available():
    # Fixed batch size for the captured graph.
    B = 1

    cuda_graph_model = make_fast_predictor().to(device).eval()

    # Static tensors — these addresses are baked into the captured graph.
    # Shapes must exactly match what the model is replayed with later.
    static_features   = torch.zeros(B, 13, device=device)
    static_vertex_pos = torch.tensor(vertice_positions, dtype=torch.float32,
                                     device=device)

    # --- Warmup before capture — must run on a side stream ---
    warmup_stream = torch.cuda.Stream()
    warmup_stream.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(warmup_stream):
        for _ in tqdm(range(1000), desc="CUDA graph warmup"):
            with torch.inference_mode():
                _ = cuda_graph_model(static_features, vertex_pos=static_vertex_pos)
    torch.cuda.current_stream().wait_stream(warmup_stream)
    torch.cuda.synchronize()

    # --- Capture ---
    cuda_graph = torch.cuda.CUDAGraph()
    with torch.inference_mode(), torch.cuda.graph(cuda_graph):
        static_output = cuda_graph_model(static_features, vertex_pos=static_vertex_pos)
    print("CUDA graph captured.")

    # --- Benchmark ---
    # Pre-generate inputs OUTSIDE the timing loop.
    inputs = [torch.rand(B, 13, device=device) for _ in range(NUM_BENCHMARKS)]
    torch.cuda.synchronize()

    cuda_graph_results = []
    with torch.inference_mode():
        for x in tqdm(inputs, desc="CUDA graph benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            static_features.copy_(x)
            cuda_graph.replay()
            torch.cuda.synchronize()
            end = time.perf_counter()
            cuda_graph_results.append((end - start) * 1000)

    print(f"\n=== CUDA Graph Benchmark Results ({device}) ===")
    print(f"Max:    {max(cuda_graph_results):.4f} ms")
    print(f"Min:    {min(cuda_graph_results):.4f} ms")
    print(f"Avg:    {sum(cuda_graph_results) / len(cuda_graph_results):.4f} ms")
    print(f"Median: {sorted(cuda_graph_results)[len(cuda_graph_results) // 2]:.4f} ms")
    print(f"\nSpeedup over baseline: "
          f"{(sum(results) / len(results)) / (sum(cuda_graph_results) / len(cuda_graph_results)):.1f}x")


CUDA graph warmup: 100%|██████████| 1000/1000 [00:01<00:00, 529.30it/s]


CUDA graph captured.


CUDA graph benchmark: 100%|██████████| 1000/1000 [00:00<00:00, 1839.03it/s]


=== CUDA Graph Benchmark Results (cuda) ===
Max:    1.2860 ms
Min:    0.4320 ms
Avg:    0.5265 ms
Median: 0.4539 ms

Speedup over baseline: 1.7x


## 4. Loss

Huber (smooth-L1) loss replaces L1.  Near zero it's quadratic (smooth gradients,
won't over-punish tiny residuals); far from zero it's linear (robust to the
occasional outlier).  `delta=1.0` is in *normalized* target units so it's
roughly one standard deviation of the target.

**Energy-conservation penalty.**  For each collision sample we integrate one
timestep forward using the *predicted* wrench and check whether the body's
kinetic energy would grow beyond `e^2 * KE_before` (where `e` is the
coefficient of restitution).  Any excess is squared and added to the loss,
so the term is zero for physically admissible predictions and grows smoothly
when the model would inject energy — the exact failure mode you were seeing
at low collision speeds.  You control it with `w_energy`, `dt`, `mass`,
`inertia_diag`, and `restitution` when constructing `WrenchLoss`.


In [20]:
class WrenchLoss(nn.Module):
    """Physics-structured loss for a Hooke (linear elastic) contact model.

    Components:
        - BCE on collision_logit (with pos_weight for class imbalance).
        - Masked Huber on force and torque, operating in PHYSICAL units.
        - Soft penalty on f_n < 0 (Signorini violation).
        - Soft penalty on energy gain during collision (restitution-aware),
          using a BOUNDED log-based term so large violations at init don't
          produce enormous gradients that kill regression learning.

    Energy-conservation term
    ------------------------
    Given a predicted force F and torque tau applied over one timestep dt to a
    body with mass m and (diagonal) inertia I, the post-step velocities are
        v'   = v   + (F   / m) * dt
        w'   = w   + (I^-1 tau) * dt
    and the kinetic energy is  KE = 0.5 m |v|^2 + 0.5 w^T I w.
    For a real collision with coefficient of restitution e in [0, 1] we expect
        KE'  <=  e^2 * KE_before.
    We define the energy ratio
        r = KE' / (e^2 * KE_before)
    and penalise log(max(r, 1))^2. This is zero when r <= 1 (admissible),
    grows like (log r)^2 when r > 1, and its gradient in r is bounded — so a
    random-init network that predicts wildly wrong forces at epoch 0 won't
    produce an exploding energy gradient that pushes the model into the
    degenerate F≈0 basin.

    Warmup: w_energy typically starts at 0 and is ramped up over several
    epochs by the training loop via `set_energy_weight`, so the regression
    heads (force, torque) get to learn first before conservation pressure
    kicks in.

    Because targets are NOT pre-normalised, you may need to set w_force and
    w_torque to bring the two regression terms to comparable magnitude. A good
    heuristic is to set:
        w_force  ~ 1 / (force_rms_in_physical_units)
        w_torque ~ 1 / (torque_rms_in_physical_units)
    so both contribute roughly equally early in training.
    """
    def __init__(self,
                 w_force=1.0, w_torque=1.0, w_collision=1.0,
                 w_energy=0.0,
                 huber_delta=1.0, pos_weight=None,
                 dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                 restitution=0.5):
        super().__init__()
        self.w_force     = w_force
        self.w_torque    = w_torque
        self.w_collision = w_collision
        self.w_energy    = w_energy
        self.huber_delta = huber_delta
        self.dt          = float(dt)
        self.mass        = float(mass)
        self.restitution = float(restitution)
        # Inertia tensor (diagonal) for a unit cube by default: I = (1/6) m a^2
        # with m=1, a=1. Override via the constructor to match your simulated body.
        self.register_buffer(
            "inertia_diag",
            torch.tensor(inertia_diag, dtype=torch.float32),
        )
        # pos_weight is a tensor; register as buffer so .to(device) moves it.
        if pos_weight is not None and not torch.is_tensor(pos_weight):
            pos_weight = torch.tensor(float(pos_weight))
        self.register_buffer(
            "pos_weight",
            pos_weight if pos_weight is not None else torch.tensor(1.0),
        )
        self._has_pos_weight = pos_weight is not None

    def set_energy_weight(self, w):
        """Runtime hook for the training loop's warmup schedule."""
        self.w_energy = float(w)

    def _kinetic_energy(self, v, w):
        """KE = 0.5 m |v|^2 + 0.5 w^T I w for diagonal I. Shapes: (B,3)."""
        ke_lin = 0.5 * self.mass * (v * v).sum(dim=-1, keepdim=True)
        ke_rot = 0.5 * (self.inertia_diag * w * w).sum(dim=-1, keepdim=True)
        return ke_lin + ke_rot

    def forward(self, preds, targets):
        mask   = targets["is_collision"]                   # (B, 1)
        n_coll = mask.sum().clamp_min(1.0)

        # --- collision BCE ---
        loss_collision = F.binary_cross_entropy_with_logits(
            preds["collision_logit"], mask.float(),
            pos_weight=self.pos_weight if self._has_pos_weight else None,
            reduction="mean",
        )
        # --- masked Huber on force and torque (physical units) ---
        raw_f = F.huber_loss(preds["force"],  targets["force"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        raw_t = F.huber_loss(preds["torque"], targets["torque"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        loss_force  = (raw_f.sum(dim=-1, keepdim=True) * mask).sum() / n_coll
        loss_torque = (raw_t.sum(dim=-1, keepdim=True) * mask).sum() / n_coll

        # --- energy-conservation penalty (bounded, log-based) ---
        # Integrate one step using the predicted wrench and compare KE before vs after.
        # Only collision samples contribute (non-contact steps have F=tau=0 anyway).
        v = targets["lin_vel"]                              # (B, 3)
        w = targets["ang_vel"]                              # (B, 3)
        f_pred = preds["force"]                             # (B, 3)
        t_pred = preds["torque"]                            # (B, 3)

        v_next = v + (f_pred / self.mass) * self.dt
        # Diagonal inertia -> element-wise divide
        w_next = w + (t_pred / self.inertia_diag) * self.dt

        ke_before = self._kinetic_energy(v, w)              # (B, 1)
        ke_after  = self._kinetic_energy(v_next, w_next)    # (B, 1)

        # Log-ratio penalty:
        #   r = ke_after / (e^2 * ke_before + eps),  penalty = max(log r, 0)^2
        # Bounded gradient in F: d/dF log(ke_after) scales as 1/ke_after, so at
        # init where ke_after is huge the gradient is SMALL — the opposite of
        # (ke_after - budget)^2, which has gradient proportional to ke_after.
        eps       = 1e-6
        ke_budget = (self.restitution ** 2) * ke_before
        log_ratio = torch.log(ke_after + eps) - torch.log(ke_budget + eps)
        excess_log  = F.relu(log_ratio)                     # zero when admissible
        loss_energy = ((excess_log ** 2) * mask).sum() / n_coll

        total = (self.w_force     * loss_force
               + self.w_torque    * loss_torque
               + self.w_collision * loss_collision
               + self.w_energy    * loss_energy)

        # Diagnostic: fraction of collision samples that currently violate conservation,
        # plus the geometric-mean ratio so you can see HOW MUCH they violate by.
        with torch.no_grad():
            violating = ((excess_log > 0).float() * mask).sum() / n_coll
            # Mean log-ratio over collision samples (in log space so it's well-behaved)
            mean_log_ratio = (log_ratio * mask).sum() / n_coll

        return total, {
            "force":             loss_force.item(),
            "torque":            loss_torque.item(),
            "collision":         loss_collision.item(),
            "energy":            loss_energy.item(),
            "energy_violating":  violating.item(),
            "energy_mean_logr":  mean_log_ratio.item(),
            "w_energy":          self.w_energy,
            "total":             total.item(),
            "active_collisions": n_coll.item(),
            "k":                 preds["aux"]["k"].item(),
        }

## Training

In [21]:
def train_model(model, train_loader, val_loader, train_dataset,
                epochs=200, lr=1e-4, weight_decay=1e-4,
                w_force=1.0, w_torque=1.0, w_collision=0.5,
                # Energy-conservation warmup schedule.
                # Rationale: starting with w_energy>0 produces enormous gradients
                # at init (ke_after is huge for a random-init network) and pushes
                # the model into the degenerate F≈0 basin, where regression loss
                # plateaus at force_rms. Warming up from 0 lets the force/torque
                # heads learn a reasonable solution first; only then do we
                # gently tighten conservation.
                w_energy_max=0.1, energy_warmup_start=20, energy_warmup_epochs=30,
                dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                restitution=0.5):
    model.to(device)

    # Vertex positions are constant across all samples — build once.
    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

    # Class-imbalance weight for BCE: #no-contact / #contact on train set.
    collisions = train_dataset.collisions.squeeze(-1).bool()
    n_pos = int(collisions.sum().item())
    n_neg = int((~collisions).sum().item())
    pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0
    print(f"BCE pos_weight = {pos_weight:.3f}  ({n_pos} contacts / {n_neg} non-contacts)")

    criterion = WrenchLoss(
        w_force=w_force, w_torque=w_torque, w_collision=w_collision,
        w_energy=0.0,  # ramped up by the warmup schedule below
        dt=dt, mass=mass, inertia_diag=inertia_diag, restitution=restitution,
        huber_delta=1.0, pos_weight=pos_weight,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=20
    )

    def energy_weight_at(epoch):
        """Linear ramp from 0 to w_energy_max over [start, start+epochs)."""
        if epoch < energy_warmup_start:
            return 0.0
        if energy_warmup_epochs <= 0:
            return float(w_energy_max)
        frac = (epoch - energy_warmup_start) / float(energy_warmup_epochs)
        return float(w_energy_max) * min(max(frac, 0.0), 1.0)

    best_val_loss = float('inf')
    loss_keys = ['total', 'force', 'torque', 'collision',
                 'energy', 'energy_violating', 'energy_mean_logr']

    for epoch in tqdm(range(epochs)):
        criterion.set_energy_weight(energy_weight_at(epoch))

        # --- Train ---
        model.train()
        train_losses = {k: 0.0 for k in loss_keys}
        for features, targets in train_loader:
            features = features.to(device)                       # [B, 13]
            targets  = {k: v.to(device) for k, v in targets.items()}
            out = model(
                features,
                vertex_pos=vp,
                body_position=targets["self_position"],
            )
            optimizer.zero_grad()

            loss, components = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            for k in loss_keys:
                train_losses[k] += components[k]
        for k in loss_keys:
            train_losses[k] /= len(train_loader)

        # --- Validate ---
        model.eval()
        val_losses = {k: 0.0 for k in loss_keys}
        with torch.no_grad():
            for features, targets in val_loader:
                features = features.to(device)
                targets  = {k: v.to(device) for k, v in targets.items()}
                _, components = criterion(
                    model(
                        features,
                        vertex_pos=vp,
                        body_position=targets["self_position"],
                    ),
                    targets,
                )
                for k in loss_keys:
                    val_losses[k] += components[k]
        for k in loss_keys:
            val_losses[k] /= len(val_loader)

        scheduler.step(val_losses['total'])

        if val_losses['total'] < best_val_loss:
            best_val_loss = val_losses['total']
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss':        best_val_loss,
            }, 'wrench_model_best.pth')

        if (epoch + 1) % 10 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            k_now  = components.get('k', float('nan'))
            we_now = criterion.w_energy
            print(f"Epoch {epoch+1}/{epochs} | LR: {lr_now:.2e} | k={k_now:.1f} | w_energy={we_now:.4f}")
            print(f"  Train: total={train_losses['total']:.4f} "
                  f"f={train_losses['force']:.4f} t={train_losses['torque']:.4f} "
                  f"c={train_losses['collision']:.4f} "
                  f"e={train_losses['energy']:.4f} vio={train_losses['energy_violating']:.2f} "
                  f"logr={train_losses['energy_mean_logr']:+.3f}")
            print(f"  Val:   total={val_losses['total']:.4f} "
                  f"f={val_losses['force']:.4f} t={val_losses['torque']:.4f} "
                  f"c={val_losses['collision']:.4f} "
                  f"e={val_losses['energy']:.4f} vio={val_losses['energy_violating']:.2f} "
                  f"logr={val_losses['energy_mean_logr']:+.3f}")

    return model


### Execute training

In [22]:
model = train_model(model, train_loader, val_loader, full_dataset,
                    epochs=200, lr=1e-3,
                    w_force=1.0, w_torque=1.0, w_collision=0.5)


BCE pos_weight = 0.912  (1024037 contacts / 934025 non-contacts)


  0%|          | 0/200 [00:00<?, ?it/s]

  5%|▌         | 10/200 [13:21<4:11:59, 79.58s/it]

Epoch 10/200 | LR: 1.00e-03 | k=996.3 | w_energy=0.0000
  Train: total=429.6783 f=376.3773 t=53.2031 c=0.1959 e=28.3835 vio=1.00 logr=+4.878
  Val:   total=432.2695 f=372.5358 t=59.6317 c=0.2042 e=28.2962 vio=1.00 logr=+4.881


 10%|█         | 20/200 [27:16<4:08:47, 82.93s/it]

...

Epoch 180/200 | LR: 1.00e-03 | k=963.8 | w_energy=0.1000
  Train: total=86.9823 f=59.0685 t=24.7950 c=0.0612 e=30.8818 vio=1.00 logr=+5.001
  Val:   total=121.5658 f=90.4325 t=27.8829 c=0.0635 e=32.1861 vio=1.00 logr=+5.114


 95%|█████████▌| 190/200 [4:41:25<14:07, 84.80s/it]

Epoch 190/200 | LR: 5.00e-04 | k=962.7 | w_energy=0.1000
  Train: total=61.8559 f=42.4315 t=16.3095 c=0.0516 e=30.8900 vio=1.00 logr=+4.999
  Val:   total=94.7720 f=68.2276 t=23.4345 c=0.0480 e=30.8587 vio=1.00 logr=+4.996


100%|██████████| 200/200 [4:56:51<00:00, 89.06s/it]

Epoch 200/200 | LR: 5.00e-04 | k=961.8 | w_energy=0.1000
  Train: total=60.8586 f=41.7405 t=16.0018 c=0.0505 e=30.9114 vio=1.00 logr=+5.001
  Val:   total=97.0749 f=71.7182 t=22.2728 c=0.0483 e=30.5978 vio=1.00 logr=+4.975


## Evaluation

In [23]:
@torch.no_grad()
def evaluate(model, loader, dataset, device="cuda",
             rel_floor_force=0.05, rel_floor_torque=0.05):
    """Evaluate in physical units (targets were not normalised).

    Args:
        dataset: the *underlying* ContactDataset (not a Subset). Kept for API
                 symmetry; no target stats are needed since targets are physical.
        rel_floor_{force,torque}: targets with magnitude below this (in
                 physical units) are excluded from the relative-error stats
                 to avoid division-by-near-zero blow-up.
    """
    model.eval()
    all_abs_f, all_abs_t = [], []
    all_rel_f, all_rel_t = [], []
    all_coll_correct     = []

    # Physics-diagnostic accumulators (frictionless model: only f_n sign check)
    n_fn_neg = 0
    n_total  = 0

    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

    for features, targets in loader:
        features = features.to(device)
        f_tgt = targets["force"].to(device)
        t_tgt = targets["torque"].to(device)
        c_tgt = targets["is_collision"].to(device).squeeze(-1).bool()
        body_position = targets["self_position"].to(device)

        preds = model(features, vertex_pos=vp, body_position=body_position)

        f_pred = preds["force"]
        t_pred = preds["torque"]
        c_pred = (torch.sigmoid(preds["collision_logit"]).squeeze(-1) > 0.5)

        all_coll_correct.append((c_pred == c_tgt).float().cpu())

        if c_tgt.any():
            f_pred_c = f_pred[c_tgt]
            f_tgt_c  = f_tgt[c_tgt]
            t_pred_c = t_pred[c_tgt]
            t_tgt_c  = t_tgt[c_tgt]

            abs_f = (f_pred_c - f_tgt_c).norm(dim=-1)
            abs_t = (t_pred_c - t_tgt_c).norm(dim=-1)
            all_abs_f.append(abs_f.cpu())
            all_abs_t.append(abs_t.cpu())

            f_norm = f_tgt_c.norm(dim=-1)
            t_norm = t_tgt_c.norm(dim=-1)
            mask_f = f_norm > rel_floor_force
            mask_t = t_norm > rel_floor_torque
            if mask_f.any():
                rel_f = (f_pred_c[mask_f] - f_tgt_c[mask_f]).norm(dim=-1) / f_norm[mask_f]
                all_rel_f.append(rel_f.cpu())
            if mask_t.any():
                rel_t = (t_pred_c[mask_t] - t_tgt_c[mask_t]).norm(dim=-1) / t_norm[mask_t]
                all_rel_t.append(rel_t.cpu())

            # Physics diagnostic: how often the Hooke-plus-residual allows f_n < 0.
            f_n_c = preds["aux"]["force_residual"][c_tgt]
            n_fn_neg += int((f_n_c < 0).sum().item())
            n_total  += int(c_tgt.sum().item())

    abs_f = torch.cat(all_abs_f) if all_abs_f else torch.empty(0)
    abs_t = torch.cat(all_abs_t) if all_abs_t else torch.empty(0)
    rel_f = torch.cat(all_rel_f) if all_rel_f else torch.empty(0)
    rel_t = torch.cat(all_rel_t) if all_rel_t else torch.empty(0)
    coll_acc = torch.cat(all_coll_correct).mean().item()

    def fmt_pct(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.1%}  p90={e.quantile(0.9):.1%}  "
                f"p99={e.quantile(0.99):.1%}")
    def fmt_abs(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.4f}  p90={e.quantile(0.9):.4f}  "
                f"p99={e.quantile(0.99):.4f}")

    print("─" * 60)
    print(f"Collision accuracy : {coll_acc:.3%}")
    print(f"Force  abs err     : {fmt_abs(abs_f)}")
    print(f"Torque abs err     : {fmt_abs(abs_t)}")
    print(f"Force  rel err     : {fmt_pct(rel_f)}  "
          f"(on {rel_f.numel()} / {abs_f.numel()} samples above floor)")
    print(f"Torque rel err     : {fmt_pct(rel_t)}  "
          f"(on {rel_t.numel()} / {abs_t.numel()} samples above floor)")
    if n_total > 0:
        print(f"Physics violations : f_n<0 in {n_fn_neg}/{n_total} "
              f"({100*n_fn_neg/n_total:.2f}%)")
    print("─" * 60)

    return {
        "collision_acc":  coll_acc,
        "force_abs_p50":  abs_f.median().item() if abs_f.numel() else float("nan"),
        "force_abs_p99":  abs_f.quantile(0.99).item() if abs_f.numel() else float("nan"),
        "torque_abs_p50": abs_t.median().item() if abs_t.numel() else float("nan"),
        "torque_abs_p99": abs_t.quantile(0.99).item() if abs_t.numel() else float("nan"),
        "fn_neg_rate":    (n_fn_neg / n_total) if n_total > 0 else float("nan"),
    }


## Print 100 data points

In [24]:
@torch.no_grad()
def print_predictions(model, loader, n=100):
    """Print n predictions vs ground truth from the loader."""
    model.eval()
    all_f_pred, all_f_tgt = [], []
    all_t_pred, all_t_tgt = [], []
    all_coll_pred, all_coll_tgt = [], []

    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

    for features, targets in loader:
        features = features.to(device)
        body_position = targets["self_position"].to(device)
        preds = model(features, vertex_pos=vp, body_position=body_position)

        all_f_pred.append(preds["force"].cpu())
        all_t_pred.append(preds["torque"].cpu())
        all_f_tgt.append(targets["force"])
        all_t_tgt.append(targets["torque"])
        all_coll_pred.append(torch.sigmoid(preds["collision_logit"]).cpu())
        all_coll_tgt.append(targets["is_collision"])
        collected = sum(x.shape[0] for x in all_f_pred)
        if collected >= n:
            break

    f_pred = torch.cat(all_f_pred)[:n]
    f_tgt  = torch.cat(all_f_tgt)[:n]
    t_pred = torch.cat(all_t_pred)[:n]
    t_tgt  = torch.cat(all_t_tgt)[:n]
    c_pred = torch.cat(all_coll_pred)[:n].squeeze(-1)
    c_tgt  = torch.cat(all_coll_tgt)[:n].squeeze(-1)

    header = (f"{'#':>4s}  {'coll':>5s} {'pred':>5s}  "
              f"{'force_pred':>30s}  {'force_true':>30s}  "
              f"{'torque_pred':>30s}  {'torque_true':>30s}  ")
    print(header)
    print("─" * len(header))
    for i in range(n):
        cp = f"{c_pred[i]:.2f}"
        ct = f"{int(c_tgt[i].item())}"
        fp = f"[{f_pred[i,0]:8.3f}, {f_pred[i,1]:8.3f}, {f_pred[i,2]:8.3f}]"
        ft = f"[{f_tgt[i,0]:8.3f}, {f_tgt[i,1]:8.3f}, {f_tgt[i,2]:8.3f}]"
        tp = f"[{t_pred[i,0]:8.4f}, {t_pred[i,1]:8.4f}, {t_pred[i,2]:8.4f}]"
        tt = f"[{t_tgt[i,0]:8.4f}, {t_tgt[i,1]:8.4f}, {t_tgt[i,2]:8.4f}]"
        print(f"{i:4d}  {ct:>5s} {cp:>5s}  {fp:>30s}  {ft:>30s}  {tp:>30s}  {tt:>30s}")


print_predictions(model, val_loader, n=100)


   #   coll  pred                      force_pred                      force_true                     torque_pred                     torque_true  
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   0      1  1.00  [   0.018,   -0.015,   79.422]  [   0.000,   -0.000,   76.688]  [ -9.5557,   1.3330,   0.0024]  [ -9.1401,   1.8032,   0.0000]
   1      1  0.99  [   0.003,   -0.017,   87.063]  [  -0.000,    0.000,   88.626]  [ 32.3935,  38.2566,   0.0063]  [ 33.1813,  39.5477,  -0.0000]
   2      0  0.01  [   0.035,   -0.024,  250.491]  [   0.000,    0.000,    0.000]  [ 39.0076,  23.8223,  -0.0031]  [  0.0000,   0.0000,   0.0000]
   3      0  0.00  [   0.057,   -0.071,  143.339]  [   0.000,    0.000,    0.000]  [ 12.9611,  11.7450,   0.0007]  [  0.0000,   0.0000,   0.0000]
   4      1  1.00  [  -0.023,   -0.043,  432.758]  [   0.000,    0.000,  435.024]  [-104.2706, 195.9004,   0.0138]  [-10

Download checkpoint (Colab)

In [25]:
if is_colab():
    from google.colab import files
    files.download("wrench_model_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>